#  AIOps en Redes: Diagnóstico Inteligente con LLMs
**Workshop universitario - Panamá**

Los equipos de redes reciben alertas constantemente pero no siempre tienen capacidad de
diagnóstico inmediato. Este notebook automatiza ese proceso:
**detecta, visualiza, diagnostica y reporta** en menos de un minuto.

---

| Paso | El sistema... | Herramienta |
|---|---|---|
| 1 | Detecta anomalias en el tráfico | Umbrales operacionales |
| 2 | Visualiza las métricas clave | Matplotlib / Seaborn |
| 3 | Diagnostica el problema | LLM Qwen2.5-1.5B |
| 4 | Reporta a gerencia | LLM (tono ejecutivo) |

---

> **Tiempo estimado:** 30-45 minutos | **Requisitos:** Google Colab + GPU T4 (gratis)

## Paso 1 - El sistema detecta

Antes de graficar cualquier cosa, el sistema evalúa el tráfico contra umbrales operacionales:

| Umbral | Valor | Qué indica |
|---|---|---|
| `UMBRAL_PACKET_LOSS` | 5% | Pérdida de paquetes anormal |
| `UMBRAL_LATENCIA` | 200 ms | Delay elevado |
| `FACTOR_CONGESTION` | 3× baseline | Posible congestión o DDoS |
| `UMBRAL_BW_HOST` | 50% del total | Consumo anormal de un solo host |

El resultado aparece **antes** de cualquier gráfica para forzar el análisis sin sesgos visuales.

### Celda 1 - Instalación de dependencias

Instalamos las librerías necesarias para el workshop:
- `transformers` + `accelerate`: cargar modelos LLM desde Hugging Face
- `pandas` + `matplotlib` + `seaborn`: análisis y visualización de datos

> ⏳ Esta celda tarda ~2 minutos la primera vez.

In [ ]:
!pip install -q transformers accelerate pandas matplotlib seaborn
print(' Dependencias instaladas')

### Celda 2 - Cargar el dataset

Descargamos el dataset de tráfico de red sintético desde el repositorio.
Contiene flujos de red capturados durante **30 minutos** con métricas operacionales:
latencia, jitter, pérdida de paquetes, retransmisiones y profundidad de cola.

In [ ]:
import os, urllib.request
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

URL_DATOS = 'https://raw.githubusercontent.com/bocchidayo/aiops-redes-llm/main/data/trafico_red.csv'
ARCHIVO = 'trafico_red.csv'

if not os.path.exists(ARCHIVO):
    print('Descargando dataset...')
    urllib.request.urlretrieve(URL_DATOS, ARCHIVO)
    print(' Dataset descargado')
else:
    print(' Dataset ya disponible localmente')

df = pd.read_csv(ARCHIVO)
print(f'\nDimensiones : {df.shape[0]} flujos × {df.shape[1]} columnas')
print(f'Columnas    : {list(df.columns)}')
print(f'Minutos     : {df["minuto"].min()} – {df["minuto"].max()}')
df.head(10)


In [ ]:
# Umbrales operacionales
UMBRAL_PACKET_LOSS = 5.0   # %
UMBRAL_LATENCIA    = 200   # ms
FACTOR_CONGESTION  = 3.0   # múltiplo sobre la baseline
UMBRAL_BW_HOST     = 0.50  # fracción del total

alertas = []

# Agrupar por minuto para análisis de series temporales
pm = df.groupby('minuto').agg(
    flujos         = ('id_flujo',         'count'),
    bytes_total    = ('bytes',            'sum'),
    latencia_media = ('latency_ms',       'mean'),
    jitter_medio   = ('jitter_ms',        'mean'),
    loss_media     = ('packet_loss_pct',  'mean'),
    retrans_total  = ('retransmissions',  'sum'),
    queue_media    = ('queue_depth',      'mean'),
    packets_total  = ('packets',          'sum'),
).reset_index()

# Baseline: primeros 10 minutos
baseline_bytes = pm[pm['minuto'] <= 10]['bytes_total'].mean()
baseline_loss  = pm[pm['minuto'] <= 10]['loss_media'].mean()
baseline_lat   = pm[pm['minuto'] <= 10]['latencia_media'].mean()

# Detección
minutos_high_loss = pm[pm['loss_media'] > UMBRAL_PACKET_LOSS]
if not minutos_high_loss.empty:
    mins = minutos_high_loss['minuto'].tolist()
    alertas.append(f'  Pérdida de paquetes detectada - minutos {mins} (>{UMBRAL_PACKET_LOSS}%)')

minutos_alto_delay = pm[pm['latencia_media'] > UMBRAL_LATENCIA]
if not minutos_alto_delay.empty:
    mins = minutos_alto_delay['minuto'].tolist()
    alertas.append(f'  Alto delay detectado - minutos {mins} (>{UMBRAL_LATENCIA} ms)')

minutos_congestion = pm[pm['bytes_total'] > baseline_bytes * FACTOR_CONGESTION]
if not minutos_congestion.empty:
    mins = minutos_congestion['minuto'].tolist()
    alertas.append(f'  Posible congestión - minutos {mins} (>{FACTOR_CONGESTION}× baseline)')

bytes_por_host = df.groupby('src_ip')['bytes'].sum()
total_bytes = bytes_por_host.sum()
host_dominante = bytes_por_host.idxmax()
fraccion_host = bytes_por_host.max() / total_bytes
if fraccion_host > UMBRAL_BW_HOST:
    alertas.append(f'  Consumo anormal de ancho de banda - {host_dominante} consume {fraccion_host*100:.1f}% del total')

# Resultado
print('=' * 55)
print('   SISTEMA DE ALERTAS AUTOMÁTICAS - NOC')
print('=' * 55)
if alertas:
    for a in alertas:
        print(a)
    print(f'\nSe detectaron {len(alertas)} condiciones anómalas.')
else:
    print(' Red operando dentro de parámetros normales')
print('=' * 55)


## Paso 2 - El sistema visualiza

Dashboard de 4 métricas operacionales por minuto:
- **Throughput** (MB/min): volumen de tráfico total
- **Latencia promedio** (ms): delay medio de los flujos
- **Pérdida de paquetes** (%): porcentaje de pérdida
- **Top 5 hosts** por ancho de banda consumido

La región roja marca automáticamente la ventana donde se detectó la mayor anomalía.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

sns.set_theme(style='darkgrid', palette='muted')

# Detectar ventana de anomalía automáticamente
minuto_anomalia = pm.loc[pm['latencia_media'].idxmax(), 'minuto']
# Si latencia no destaca, buscar por bytes
if pm.loc[pm['latencia_media'].idxmax(), 'latencia_media'] < UMBRAL_LATENCIA:
    minuto_anomalia = pm.loc[pm['bytes_total'].idxmax(), 'minuto']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Métricas de Red por Minuto - Red Corporativa', fontsize=14, fontweight='bold')

# Throughput
ax = axes[0, 0]
ax.plot(pm['minuto'], pm['bytes_total']/1e6, color='steelblue', lw=2, marker='o', ms=3)
ax.set_ylabel('MB / minuto')
ax.set_title('Throughput')
ax.axvspan(minuto_anomalia-0.5, minuto_anomalia+0.5, color='red', alpha=0.25, label='Ventana anómala')
ax.legend(fontsize=8)

# Latencia - solo flujos TCP
latencia_tcp = (df[df['protocolo'] == 'TCP']
                .groupby('minuto')['latency_ms']
                .mean()
                .reindex(pm['minuto'])
                .values)
ax = axes[0, 1]
ax.plot(pm['minuto'], latencia_tcp, color='darkorange', lw=2, marker='o', ms=3)
ax.set_ylabel('ms')
ax.set_title('Latencia promedio (TCP)')
ax.axhline(UMBRAL_LATENCIA, color='red', ls='--', lw=1.5, label=f'Umbral {UMBRAL_LATENCIA}ms')
ax.axvspan(minuto_anomalia-0.5, minuto_anomalia+0.5, color='red', alpha=0.25)
ax.legend(fontsize=8)

# Pérdida de paquetes
ax = axes[1, 0]
ax.plot(pm['minuto'], pm['loss_media'], color='crimson', lw=2, marker='o', ms=3)
ax.set_ylabel('%')
ax.set_title('Pérdida de paquetes (%)')
ax.axhline(UMBRAL_PACKET_LOSS, color='red', ls='--', lw=1.5, label=f'Umbral {UMBRAL_PACKET_LOSS}%')
ax.axvspan(minuto_anomalia-0.5, minuto_anomalia+0.5, color='red', alpha=0.25)
ax.legend(fontsize=8)

# Top 5 talkers
ax = axes[1, 1]
top5_hosts = df.groupby('src_ip')['bytes'].sum().nlargest(5)
top5_hosts_mb = top5_hosts / 1e6
ax.barh(range(len(top5_hosts_mb)), top5_hosts_mb.values, color='mediumpurple', alpha=0.85)
ax.set_yticks(range(len(top5_hosts_mb)))
ax.set_yticklabels(top5_hosts_mb.index, fontsize=9)
ax.set_xlabel('MB totales')
ax.set_title('Top 5 hosts por ancho de banda')

plt.tight_layout()
plt.savefig('trafico_red.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n La región roja marca la ventana donde se detectó la anomalía.')


### Celda 5 - Análisis estadístico

Comparamos tres ventanas temporales para cuantificar el impacto de la anomalía:
- **Baseline**: minutos 1–10 (comportamiento normal)
- **Ventana anómala**: ±2 minutos alrededor del pico detectado
- **Recuperación**: minutos posteriores a la anomalía

La tabla muestra latencia, pérdida de paquetes, throughput y bytes pico en cada ventana.

In [ ]:
# Baseline: minutos 1-10 | Anomalía: ±2 minutos alrededor del pico | Recuperación: resto
ventana_base  = pm[pm['minuto'].between(1, 10)]
ventana_anom  = pm[pm['minuto'].between(minuto_anomalia-2, minuto_anomalia+2)]
ventana_recup = pm[pm['minuto'] > minuto_anomalia+2]

def resumen(v):
    return {
        'Latencia media (ms)': round(v['latencia_media'].mean(), 1),
        'Pérdida paquetes (%)': round(v['loss_media'].mean(), 2),
        'Throughput (MB/min)': round(v['bytes_total'].mean()/1e6, 2),
        'Bytes pico (MB)': round(v['bytes_total'].max()/1e6, 2),
    }

tabla = pd.DataFrame({
    'Baseline (min 1-10)': resumen(ventana_base),
    'Ventana anomalía': resumen(ventana_anom),
    'Recuperación': resumen(ventana_recup),
})

print('=== ANÁLISIS ESTADÍSTICO ===')
print()
print(tabla.to_string())

print('\n Interpretación:')
lat_delta  = resumen(ventana_anom)['Latencia media (ms)'] - resumen(ventana_base)['Latencia media (ms)']
loss_delta = resumen(ventana_anom)['Pérdida paquetes (%)'] - resumen(ventana_base)['Pérdida paquetes (%)']
bw_delta   = resumen(ventana_anom)['Throughput (MB/min)'] / max(resumen(ventana_base)['Throughput (MB/min)'], 0.01)
print(f'  • La latencia aumentó {lat_delta:+.1f} ms durante la ventana anómala.')
print(f'  • La pérdida de paquetes aumentó {loss_delta:+.2f}% sobre la baseline.')
print(f'  • El throughput fue {bw_delta:.1f}× la baseline durante la anomalía.')


## Paso 3 - El sistema diagnostica

Aquí hacemos dos cosas:
1. Construimos un **resumen textual** con los números clave del análisis anterior
2. Definimos el **prompt** que le daremos al LLM, instruyéndolo a actuar como ingeniero NOC

El prompt le pide al modelo: diagnóstico, clasificación del problema, host probable y acción inmediata.

In [ ]:
import textwrap

# Construir resumen estadístico para el LLM
top5_src = (df.groupby('src_ip')['bytes'].sum().nlargest(5)
            .reset_index().assign(MB=lambda d: (d['bytes']/1e6).round(2))[['src_ip', 'MB']]
            .to_string(index=False))

RESUMEN_RED = textwrap.dedent(f"""
=== RESUMEN DE RED - 30 MINUTOS ===
Flujos totales     : {len(df):,}
Latencia baseline  : {resumen(ventana_base)['Latencia media (ms)']:.1f} ms
Latencia ventana   : {resumen(ventana_anom)['Latencia media (ms)']:.1f} ms
Pérdida baseline   : {resumen(ventana_base)['Pérdida paquetes (%)']:.2f}%
Pérdida ventana    : {resumen(ventana_anom)['Pérdida paquetes (%)']:.2f}%
Throughput baseline: {resumen(ventana_base)['Throughput (MB/min)']:.2f} MB/min
Throughput pico    : {resumen(ventana_anom)['Bytes pico (MB)']:.2f} MB/min
Minuto con pico    : {minuto_anomalia}
Top 5 hosts por bytes enviados:
{top5_src}
Alertas detectadas:
{chr(10).join(alertas) if alertas else 'Ninguna'}
""").strip()

SYSTEM_NOC = (
    "Eres un ingeniero de NOC (Network Operations Center) con experiencia en "
    "análisis de rendimiento de redes, SLA y resolución de incidencias. "
    "Responde siempre en español. Sé técnico, conciso y orientado a la acción."
)

USER_NOC = textwrap.dedent(f"""
Analiza este resumen de tráfico de red capturado durante 30 minutos.
Determina si hay degradación del servicio e indica:

1. **Diagnóstico**: ¿hay degradación? ¿en qué periodo y con qué severidad?
2. **Clasificación**: ¿congestión / pérdida de paquetes / consumo exagerado de BW?
3. **Host origen probable**: ¿qué IP o segmento podría ser el causante?
4. **Acción inmediata**: ¿qué debe hacer el equipo NOC ahora mismo?

--- DATOS ---
{RESUMEN_RED}
--- FIN ---

Sé específico con los números del resumen para justificar tu análisis.
""").strip()

print('=== RESUMEN QUE SE ENVIARÁ AL MODELO ===')
print()
print(RESUMEN_RED)
print()
print(f'Longitud del prompt completo: {len(USER_NOC)} caracteres')


### Celda 7 - Cargar el modelo Qwen2.5-1.5B

Descargamos y cargamos el modelo de lenguaje **Qwen2.5-1.5B-Instruct** desde Hugging Face.

- Tamaño: ~3 GB (se descarga solo la primera vez)
- Se ejecuta en la GPU T4 de Colab con precisión `float16`
- `device_map='auto'` coloca el modelo automáticamente en GPU o CPU según disponibilidad

> ⏳ Esta celda tarda **3–5 minutos** la primera vez (descarga del modelo).

In [ ]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

MODELO_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

print(f'Cargando {MODELO_ID}...')
print(f'GPU disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM total     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)
modelo = AutoModelForCausalLM.from_pretrained(
    MODELO_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)
pipe_noc = pipeline('text-generation', model=modelo, tokenizer=tokenizer)

print(f'\n Modelo cargado en {next(modelo.parameters()).device}')


### Celda 8 - Ejecutar el análisis NOC con el LLM

Enviamos el resumen estadístico al modelo y obtenemos su diagnóstico.

- `temperature=0.3`: respuestas más deterministas (ideal para análisis técnico)
- `max_new_tokens=700`: limita la longitud de la respuesta
- La respuesta se guarda en `respuesta_1b.txt` para comparar después con el modelo grande del instructor

In [ ]:
RESPUESTA_1B = '(sin respuesta)'

try:
    inicio = time.time()
    resultado = pipe_noc(
        [{'role': 'system', 'content': SYSTEM_NOC}, {'role': 'user', 'content': USER_NOC}],
        max_new_tokens=1500,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    elapsed = time.time() - inicio
    RESPUESTA_1B = resultado[0]['generated_text'][-1]['content']

    print(' Diagnóstico del modelo (Qwen2.5-1.5B):')
    print('=' * 55)
    print(RESPUESTA_1B)
    print('=' * 55)
    print(f'\nTiempo de respuesta: {elapsed:.1f}s')

    with open('respuesta_1b.txt', 'w', encoding='utf-8') as f:
        f.write(RESPUESTA_1B)
    print(' Guardado en respuesta_1b.txt')

except Exception as e:
    print(f'\n  Error al ejecutar el modelo: {e}')
    print('Verifica que la Celda 7 se ejecutó correctamente.')


### Celda 9 - Chat libre con el modelo

Ahora puedes hacerle preguntas directamente al modelo sobre la red analizada.
El modelo recibe automáticamente el resumen estadístico como contexto.

Escribe tu pregunta y presiona Enter. Deja en blanco para salir.

In [ ]:
#  Hazle tus propias preguntas sobre la red
# El modelo recibe el resumen de la red como contexto automáticamente.
#
# Preguntas sugeridas:
#   "¿Por qué hay tanto delay en el minuto 22?"
#   "¿Qué host está consumiendo más ancho de banda?"
#   "¿Esto es congestión o un consumo exagerado de BW?"
#   "¿Qué le dirías al cliente afectado por la degradación?"

print(' Chat con el modelo NOC - escribe una pregunta o deja vacío para salir.\n')

if pipe_noc is None:
    print('  Modelo no disponible. Ejecuta primero la Celda 7 con GPU habilitada.')
else:
    historial = [
        {'role': 'system', 'content': SYSTEM_NOC + f'\n\nContexto de red:\n{RESUMEN_RED}'}
    ]

    while True:
        pregunta = input('Tú: ').strip()
        if not pregunta:
            print('Saliendo del chat.')
            break

        historial.append({'role': 'user', 'content': pregunta})

        try:
            out = pipe_noc(historial, max_new_tokens=1000, temperature=0.4,
                           do_sample=True, pad_token_id=tokenizer.eos_token_id)
            respuesta = out[0]['generated_text'][-1]['content']
            historial.append({'role': 'assistant', 'content': respuesta})
            print(f'\n Modelo: {respuesta}\n')

        except Exception as e:
            print(f'Error: {e}')
            break

### Reporte ejecutivo

El modelo genera un segundo análisis orientado a **gerencia**, no a operaciones técnicas.
Sin términos de red, sin números crudos - solo el problema, el impacto y la acción recomendada.

> **Nota:** Esta celda reutiliza el modelo ya cargado en la Celda 7. No necesitas descargarlo de nuevo.

In [ ]:
REPORTE_EJECUTIVO = '(sin reporte)'

try:
    SYSTEM_REPORTE = (
        "Eres un consultor de tecnología que presenta incidencias de red a gerencia. "
        "Explica el problema en términos de impacto de negocio, sin jerga técnica. "
        "Responde en español. Usa un tono formal y ejecutivo. Máximo 3 párrafos."
    )

    # Usar el diagnóstico técnico si está disponible; si no, el resumen de red
    contenido = RESPUESTA_1B if RESPUESTA_1B != '(sin respuesta)' else RESUMEN_RED

    USER_REPORTE = (
        "Basándote en este diagnóstico técnico de red, redacta un reporte ejecutivo "
        f"para la gerencia:\n\n{contenido}"
    )

    inicio = time.time()
    resultado = pipe_noc(
        [{'role': 'system', 'content': SYSTEM_REPORTE},
         {'role': 'user',   'content': USER_REPORTE}],
        max_new_tokens=600,
        temperature=0.2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    elapsed = time.time() - inicio
    REPORTE_EJECUTIVO = resultado[0]['generated_text'][-1]['content']

    print('=' * 55)
    print('   REPORTE EJECUTIVO')
    print('=' * 55)
    print(REPORTE_EJECUTIVO)
    print('=' * 55)
    print(f'\nTiempo de respuesta: {elapsed:.1f}s')

    with open('reporte_ejecutivo.txt', 'w', encoding='utf-8') as f:
        f.write(REPORTE_EJECUTIVO)
    print(' Guardado en reporte_ejecutivo.txt')

except Exception as e:
    print(f'Error: {e}')
    print('Verifica que las celdas 7 y 8 se ejecutaron correctamente.')

## Paso 4 - El sistema reporta

Preguntas para discutir en grupo una vez que el pipeline completo haya corrido.

In [ ]:
preguntas_reflexion = [
    '1. El sistema detectó alertas automáticas antes de mostrar las gráficas - ¿coincide lo que '
       'ves en el dashboard con las alertas del Paso 1?',
    '2. La latencia se calculó solo para flujos TCP - ¿por qué tiene más sentido que '
       'promediar todos los protocolos juntos?',
    '3. Compara respuesta_1b.txt con reporte_ejecutivo.txt - ¿qué información se perdió en la '
       'traducción al lenguaje ejecutivo y qué se ganó?',
    '4. ¿Qué cambiarías en el prompt de la Celda 6 para que el diagnóstico mencione '
       'explícitamente el host responsable y el minuto exacto del pico?',
]

print('=== PREGUNTAS DE REFLEXIÓN PARA EL GRUPO ===')
print()
for p in preguntas_reflexion:
    print(p)

print('\n=== PRÓXIMOS PASOS ===')
print('• Modifica el prompt en la Celda 6 para pedir más detalle o cambiar el tono.')
print('• Experimenta con temperature= para ver cómo varía la respuesta.')
print('• Agrega métricas al resumen: retransmisiones, queue_depth por segmento.')
print('• Conecta con un sistema de monitoreo real (Grafana, Zabbix) para feeds en vivo.')
print()
print(' Repositorio : https://github.com/bocchidayo/aiops-redes-llm')
print(' Abrir en Colab: https://colab.research.google.com/github/bocchidayo/aiops-redes-llm/blob/main/notebook/aiops_workshop.ipynb')
print()
print('*Workshop AIOps en Redes - Panamá*')

---
## Challenge: Tráfico Real

Sección opcional. Reutiliza el modelo ya cargado en la Celda 7.
No se necesita código adicional - solo curiosidad.

In [ ]:
import os, sys, urllib.request, subprocess

SUMMARY_FILE  = 'data/bigflows_summary.txt'
PCAP_FILE     = 'captures/bigFlows.pcap'
SUMMARY_URL   = 'https://raw.githubusercontent.com/bocchidayo/aiops-redes-llm/main/data/bigflows_summary.txt'
PCAP_URL      = 'https://s3.amazonaws.com/tcpreplay-pcap-files/bigFlows.pcap'
SCRIPT_URL    = 'https://raw.githubusercontent.com/bocchidayo/aiops-redes-llm/main/data/pcap_to_csv.py'

os.makedirs('captures', exist_ok=True)
os.makedirs('data', exist_ok=True)

RESUMEN_PCAP = None

# Intentar cargar resumen pre-generado del repositorio (rapido)
if not os.path.exists(SUMMARY_FILE):
    try:
        urllib.request.urlretrieve(SUMMARY_URL, SUMMARY_FILE)
        print('Resumen descargado del repositorio.')
    except Exception:
        pass  # fallback: procesar PCAP

if os.path.exists(SUMMARY_FILE):
    RESUMEN_PCAP = open(SUMMARY_FILE, encoding='utf-8').read()
    print('Resumen cargado.')
else:
    # Alternativa: descargar PCAP y procesar con dpkt
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'dpkt'], check=True)
    urllib.request.urlretrieve(SCRIPT_URL, 'data/pcap_to_csv.py')

    if not os.path.exists(PCAP_FILE):
        print('Descargando bigFlows.pcap (~368 MB)...')
        urllib.request.urlretrieve(PCAP_URL, PCAP_FILE)
        print(f'Descarga completa ({os.path.getsize(PCAP_FILE)/1e6:.0f} MB).')

    print('Procesando captura (~1-2 min)...')
    result = subprocess.run(
        [sys.executable, 'data/pcap_to_csv.py',
         '--input', PCAP_FILE, '--output', SUMMARY_FILE],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('Error:', result.stderr[:400])
    elif os.path.exists(SUMMARY_FILE):
        RESUMEN_PCAP = open(SUMMARY_FILE, encoding='utf-8').read()

if RESUMEN_PCAP:
    print()
    print(RESUMEN_PCAP)


## Tu Misión

Tienes el resumen de tráfico real de una red privada.
El modelo no sabe nada sobre esta red - solo lo que tú le preguntes.

No hay respuesta correcta. El objetivo es ver hasta dónde llega el modelo con las preguntas correctas.

---

### Reconocimiento - empieza por aquí

- *¿Cuánto tiempo dura la captura y cuántos paquetes tiene en total?*
- *¿Qué protocolo domina el tráfico y tiene sentido para una red corporativa?*
- *¿Cuáles son las IPs internas más activas y cuánto tráfico generan?*

### Análisis - el núcleo del challenge

- *El puerto 5440 es el más activo con 137k paquetes - ¿qué podría estar corriendo ahí?*
- *172.16.139.250 recibe 31.6 MB pero no aparece en el top de envíos - ¿qué rol podría tener ese host?*
- *La ventana más activa tiene 95k paquetes en 30 segundos - ¿eso es normal o es un pico anómalo?*
- *Los puertos 53807, 53037 y 64953 son efímeros - ¿qué significa eso en términos de flujo de conexiones?*

### Diagnóstico NOC - cierre

- *¿Hay algún host que te llame la atención como posible problema operacional?*
- *Si tuvieras que escribir un ticket de incidente con estos datos, ¿qué pondrías?*
- *¿Qué información adicional necesitarías para determinar si hay un problema real?*

In [ ]:
if RESUMEN_PCAP is None:
    print('Ejecuta primero la Celda 11 para cargar el resumen de la captura.')
elif pipe_noc is None:
    print('Ejecuta primero la Celda 7 para cargar el modelo.')
else:
    import gc
    gc.collect()
    torch.cuda.empty_cache()

    # Omitir la seccion de perfiles de host (la mas extensa) para reducir el
    # tamano del contexto y evitar OOM en la T4 con memoria fragmentada
    lineas = RESUMEN_PCAP.split('\n')
    condensado, omitir = [], False
    for ln in lineas:
        if 'PERFIL DE HOSTS INTERNOS' in ln:
            omitir = True
        elif 'PUERTOS NO ESTANDAR' in ln:
            omitir = False
        if not omitir:
            condensado.append(ln)
    resumen_contexto = '\n'.join(condensado)

    SYSTEM_CHALLENGE = (
        "Eres un ingeniero NOC experimentado. "
        "Tienes acceso al siguiente resumen de trafico de red real. "
        "Responde las preguntas del estudiante basandote unicamente en estos datos. "
        "Si no puedes inferirlo de los datos, dilo. "
        "Responde en espanol. Se tecnico y directo.\n\n"
        f"{resumen_contexto}"
    )

    historial = [{'role': 'system', 'content': SYSTEM_CHALLENGE}]

    print('Habla con el modelo - escribe tu pregunta.')
    print('(Presiona Enter sin escribir nada para terminar)')
    print()

    while True:
        pregunta = input('Tu: ').strip()
        if not pregunta:
            break

        historial.append({'role': 'user', 'content': pregunta})

        # Conservar solo los ultimos 3 intercambios para no exceder el contexto
        mensajes = [historial[0]] + historial[-6:]

        try:
            torch.cuda.empty_cache()
            out = pipe_noc(
                mensajes,
                max_new_tokens=500,
                temperature=0.3,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
            respuesta = out[0]['generated_text'][-1]['content']
            historial.append({'role': 'assistant', 'content': respuesta})
            print(f'\nModelo: {respuesta}\n')
        except Exception as e:
            print(f'Error: {e}')
            break

    print('─── Fin del challenge ───')
    print('Que encontro tu modelo? Comparte tu diagnostico con el grupo.')
